# Exploração — airbnb.landing.listings

Notebook para análise exploratória da tabela de listings do Airbnb via Databricks Connect (serverless).

In [ ]:
from airbnb_features.common.spark_session import get_spark
from pyspark.sql import functions as F

spark = get_spark()
df = spark.read.table("airbnb.landing.listings")
print(f"Total de registros: {df.count()}")

## Schema da tabela

In [ ]:
df.printSchema()

## Primeiras linhas

In [ ]:
df.show(5, truncate=False)

## Estatísticas descritivas (colunas numéricas)

In [ ]:
numeric_cols = [f.name for f in df.schema.fields if str(f.dataType) in ("IntegerType()", "LongType()", "DoubleType()", "FloatType()")]
df.select(numeric_cols).describe().show()

## Valores nulos por coluna

In [ ]:
null_counts = df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])
null_counts.show(truncate=False)

## Distribuição por room_type

In [ ]:
df.groupBy("room_type").agg(
    F.count("id").alias("total_listings"),
    F.avg("number_of_reviews").alias("avg_reviews"),
).orderBy(F.desc("total_listings")).show()

## Top 10 bairros com mais listings

In [ ]:
df.groupBy("neighbourhood").agg(
    F.count("id").alias("total_listings"),
    F.avg("number_of_reviews").alias("avg_reviews"),
    F.avg("availability_365").alias("avg_availability_365"),
).orderBy(F.desc("total_listings")).show(10)

## Distribuição de superhosts

In [ ]:
df.groupBy("host_is_superhost").agg(
    F.count("id").alias("total_listings"),
    F.avg("number_of_reviews").alias("avg_reviews"),
    F.avg("review_scores_rating").alias("avg_rating"),
).show()

## Hosts com mais listings (top 10)

In [ ]:
df.groupBy("host_id", "host_name").agg(
    F.count("id").alias("total_listings"),
    F.sum("number_of_reviews").alias("total_reviews"),
).orderBy(F.desc("total_listings")).show(10)

## Disponibilidade média por room_type

In [ ]:
df.groupBy("room_type").agg(
    F.avg("availability_30").alias("avg_availability_30d"),
    F.avg("availability_60").alias("avg_availability_60d"),
    F.avg("availability_90").alias("avg_availability_90d"),
    F.avg("availability_365").alias("avg_availability_365d"),
).show()

## Scores de review (média geral)

In [ ]:
review_cols = [c for c in df.columns if c.startswith("review_scores_")]
df.select([F.avg(c).alias(c) for c in review_cols]).show(truncate=False)